# Ahora si el modelo de red neuronal MLP para el dataset reducido con la metodología RLF de 20 atributos 

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import os
from datetime import datetime

from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# 0. CONFIGURACION DE RUTAS
# ============================================================
RUTA_PROCESADOS = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\2_datos\2_procesados"
RUTA_RESULTADOS = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\3_resultados"

ARCHIVO_DATOS = os.path.join(RUTA_PROCESADOS, "dengue_reducido_rfe_n_20_20260810_164451.xlsx")

FEATURES_FIJAS = [
    'semana_epi', 'temp_lag_9', 'temp_lag_11', 'temp_min_lag_1', 'temp_min_lag_2',
    'temp_min_lag_3', 'hum_rel_lag_1', 'prec_lag_4', 'prec_lag_6', 'vel_vi_max_lag_1',
    'vel_vi_max_lag_2', 'vel_vi_min_lag_7', 'casos_dengue_lag_1', 'casos_media_2w',
    'casos_max_2w', 'casos_tendencia_2w', 'casos_media_4w', 'casos_tendencia_4w',
    'casos_tendencia_8w', 'hum_temp_interaction'
]

TARGET = 'casos_dengue'
OBJETIVO_MAE = 3.0

os.makedirs(RUTA_RESULTADOS, exist_ok=True)

# ============================================================
# 1. CARGA DE DATOS
# ============================================================
def cargar_datos_reducidos():
    if not os.path.exists(ARCHIVO_DATOS):
        raise FileNotFoundError(f"No se encontro el archivo especificado: {ARCHIVO_DATOS}")

    print(f"\nCargando datos: {ARCHIVO_DATOS}")
    df = pd.read_excel(ARCHIVO_DATOS)
    print(f"   Filas: {len(df)} | Columnas: {len(df.columns)}")

    columna_fecha = None
    for col in df.columns:
        if 'fecha' in col.lower() or 'date' in col.lower():
            columna_fecha = col
            break

    if columna_fecha is None:
        df['fecha'] = pd.date_range(start='2021-03-28', periods=len(df), freq='W')
    else:
        df[columna_fecha] = pd.to_datetime(df[columna_fecha])
        if columna_fecha != 'fecha':
            df = df.rename(columns={columna_fecha: 'fecha'})

    if TARGET not in df.columns:
        raise ValueError(f"No se encontro la columna objetivo '{TARGET}'")

    faltantes = [f for f in FEATURES_FIJAS if f not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan features en el archivo: {faltantes}")

    df = df.sort_values('fecha').reset_index(drop=True)
    print(f"Se usaran las {len(FEATURES_FIJAS)} features especificadas")
    return df

# ============================================================
# 2. PREPARACION DE DATOS
# ============================================================
def preparar_datos(df, train_ratio=0.8):
    X = df[FEATURES_FIJAS].values
    y = df[TARGET].values.astype(float)

    n = len(df)
    corte = int(n * train_ratio)

    X_train, X_test = X[:corte], X[corte:]
    y_train, y_test = y[:corte], y[corte:]
    fechas_train = df['fecha'].iloc[:corte]
    fechas_test = df['fecha'].iloc[corte:]

    print(f"\nDivision de datos: train={len(X_train)}, test={len(X_test)}")
    print(f"Target -> media={y.mean():.2f}, std={y.std():.2f}, max={y.max():.0f}")

    return X_train, X_test, y_train, y_test, fechas_train, fechas_test

# ============================================================
# 3. UTILIDADES: transformacion log1p + evaluacion OOF temporal
# ============================================================
def construir_pipeline_mlp(hidden_layer_sizes, alpha, learning_rate_init, semilla):
    """Pipeline: RobustScaler(X) -> MLP sobre log1p(y). 
    Se maneja fuera con funciones propias."""
    return MLPRegressor(
        hidden_layer_sizes=hidden_layer_sizes,
        activation='relu',
        solver='adam',
        alpha=alpha,
        batch_size=32,
        max_iter=3000,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=30,
        random_state=semilla,
        learning_rate='adaptive',
        learning_rate_init=learning_rate_init,
        tol=1e-6,
    )


def evaluar_oof_temporal(fit_predict_fn, X, y, n_splits=5):
    """
    Evalua un modelo con validacion temporal (TimeSeriesSplit) sin fuga de datos:
    para cada fold usa solo el pasado para entrenar y predice el futuro inmediato.
    Retorna MAE OOF y el vector de predicciones OOF (NaN donde no aplica).
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    oof = np.full(len(y), np.nan)
    for train_idx, val_idx in tscv.split(X):
        pred = fit_predict_fn(X[train_idx], y[train_idx], X[val_idx])
        oof[val_idx] = pred
    mask = ~np.isnan(oof)
    mae = mean_absolute_error(y[mask], oof[mask]) if mask.sum() > 0 else np.inf
    return mae, oof


# ============================================================
# 4. BUSQUEDA DE HIPERPARAMETROS PARA LA MLP (TimeSeriesSplit)
# ============================================================
def buscar_hiperparametros_mlp(X_train, y_train, n_iter=15, n_splits=4, semilla=42):
    print(f"\n{'='*70}\nBUSQUEDA DE HIPERPARAMETROS (MLP, validacion temporal)\n{'='*70}")

    scaler_X = RobustScaler(quantile_range=(5, 95))
    X_s = scaler_X.fit_transform(X_train)
    y_log = np.log1p(y_train)

    param_dist = {
        'hidden_layer_sizes': [(128, 64, 32, 16), (100, 50, 25), (64, 32, 16),
                                (128, 64, 32), (150, 75, 30, 10)],
        'alpha': [0.0001, 0.0005, 0.001, 0.005, 0.01],
        'learning_rate_init': [0.001, 0.003, 0.005, 0.008, 0.01],
    }

    base = MLPRegressor(
        activation='relu', solver='adam', batch_size=32, max_iter=2000,
        early_stopping=True, validation_fraction=0.15, n_iter_no_change=25,
        random_state=semilla, learning_rate='adaptive', tol=1e-5
    )

    tscv = TimeSeriesSplit(n_splits=n_splits)
    busqueda = RandomizedSearchCV(
        base, param_distributions=param_dist, n_iter=n_iter, cv=tscv,
        scoring='neg_mean_absolute_error', random_state=semilla, n_jobs=-1, verbose=0
    )
    busqueda.fit(X_s, y_log)

    print(f"   Mejor MAE (log-espacio, OOF): {-busqueda.best_score_:.4f}")
    print(f"   Mejores parametros: {busqueda.best_params_}")

    return busqueda.best_params_


# ============================================================
# 5. ENSEMBLE MULTI-MODELO PONDERADO (MLP bagging + GBR + RF)
# ============================================================
def entrenar_sistema_blend(X_train, y_train, X_test, y_test, mejores_params_mlp,
                           n_mlp=15, semilla_base=100):
    print(f"\n{'='*70}\nENTRENANDO SISTEMA BLEND (MLP ensemble + GBR + RF)\n{'='*70}")

    scaler_X = RobustScaler(quantile_range=(5, 95))
    X_train_s = scaler_X.fit_transform(X_train)
    X_test_s = scaler_X.transform(X_test)
    y_train_log = np.log1p(y_train)

    # ---------- 5.1 Funciones fit-predict para evaluacion OOF ----------
    def fit_predict_mlp(Xtr, ytr_log, Xval):
        m = construir_pipeline_mlp(mejores_params_mlp['hidden_layer_sizes'],
                                    mejores_params_mlp['alpha'],
                                    mejores_params_mlp['learning_rate_init'],
                                    semilla=semilla_base)
        m.fit(Xtr, ytr_log)
        return np.expm1(np.maximum(m.predict(Xval), 0))

    def fit_predict_gbr(Xtr, ytr_log, Xval):
        m = GradientBoostingRegressor(
            n_estimators=400, max_depth=3, learning_rate=0.03,
            subsample=0.8, random_state=semilla_base
        )
        m.fit(Xtr, ytr_log)
        return np.expm1(np.maximum(m.predict(Xval), 0))

    def fit_predict_rf(Xtr, ytr_log, Xval):
        m = RandomForestRegressor(
            n_estimators=500, max_depth=8, min_samples_leaf=2,
            random_state=semilla_base, n_jobs=-1
        )
        m.fit(Xtr, ytr_log)
        return np.expm1(np.maximum(m.predict(Xval), 0))

    # ---------- 5.2 MAE OOF por modelo (para ponderar el blend) ----------
    print("Calculando desempeno OOF (validacion temporal) de cada modelo...")
    mae_mlp_oof, _ = evaluar_oof_temporal(fit_predict_mlp, X_train_s, y_train_log if False else y_train)
    # nota: pasamos y_train (escala real) a evaluar_oof_temporal para poder medir MAE real;
    # dentro de cada fit-predict se aplica log1p internamente a los datos de entrenamiento del fold
    def fit_predict_mlp_v2(Xtr, ytr, Xval):
        return fit_predict_mlp(Xtr, np.log1p(ytr), Xval)

    def fit_predict_gbr_v2(Xtr, ytr, Xval):
        return fit_predict_gbr(Xtr, np.log1p(ytr), Xval)

    def fit_predict_rf_v2(Xtr, ytr, Xval):
        return fit_predict_rf(Xtr, np.log1p(ytr), Xval)

    mae_mlp_oof, _ = evaluar_oof_temporal(fit_predict_mlp_v2, X_train_s, y_train)
    mae_gbr_oof, _ = evaluar_oof_temporal(fit_predict_gbr_v2, X_train_s, y_train)
    mae_rf_oof, _ = evaluar_oof_temporal(fit_predict_rf_v2, X_train_s, y_train)

    print(f"   MAE OOF MLP: {mae_mlp_oof:.3f}")
    print(f"   MAE OOF GBR: {mae_gbr_oof:.3f}")
    print(f"   MAE OOF RF:  {mae_rf_oof:.3f}")

    # Pesos inversamente proporcionales al MAE OOF (mejor modelo pesa mas)
    inv = np.array([1 / mae_mlp_oof, 1 / mae_gbr_oof, 1 / mae_rf_oof])
    pesos = inv / inv.sum()
    print(f"   Pesos del blend -> MLP: {pesos[0]:.3f} | GBR: {pesos[1]:.3f} | RF: {pesos[2]:.3f}")

    # ---------- 5.3 Entrenamiento final sobre todo el train ----------
    print(f"\nEntrenando ensemble final de {n_mlp} redes MLP (bagging por semilla)...")
    preds_train_mlp, preds_test_mlp = [], []
    for i in range(n_mlp):
        m = construir_pipeline_mlp(mejores_params_mlp['hidden_layer_sizes'],
                                    mejores_params_mlp['alpha'],
                                    mejores_params_mlp['learning_rate_init'],
                                    semilla=semilla_base + i)
        m.fit(X_train_s, y_train_log)
        preds_train_mlp.append(np.expm1(np.maximum(m.predict(X_train_s), 0)))
        preds_test_mlp.append(np.expm1(np.maximum(m.predict(X_test_s), 0)))
    y_train_mlp = np.median(preds_train_mlp, axis=0)
    y_test_mlp = np.median(preds_test_mlp, axis=0)

    print("Entrenando Gradient Boosting final...")
    gbr = GradientBoostingRegressor(n_estimators=400, max_depth=3, learning_rate=0.03,
                                     subsample=0.8, random_state=semilla_base)
    gbr.fit(X_train_s, y_train_log)
    y_train_gbr = np.expm1(np.maximum(gbr.predict(X_train_s), 0))
    y_test_gbr = np.expm1(np.maximum(gbr.predict(X_test_s), 0))

    print("Entrenando Random Forest final...")
    rf = RandomForestRegressor(n_estimators=500, max_depth=8, min_samples_leaf=2,
                                random_state=semilla_base, n_jobs=-1)
    rf.fit(X_train_s, y_train_log)
    y_train_rf = np.expm1(np.maximum(rf.predict(X_train_s), 0))
    y_test_rf = np.expm1(np.maximum(rf.predict(X_test_s), 0))

    # ---------- 5.4 Blend ponderado ----------
    y_train_blend = pesos[0] * y_train_mlp + pesos[1] * y_train_gbr + pesos[2] * y_train_rf
    y_test_blend = pesos[0] * y_test_mlp + pesos[1] * y_test_gbr + pesos[2] * y_test_rf

    def calc_metrics(y_true, y_pred):
        return {
            'mae': mean_absolute_error(y_true, y_pred),
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
            'r2': r2_score(y_true, y_pred)
        }

    resultados = {
        'mlp': {'train': calc_metrics(y_train, y_train_mlp), 'test': calc_metrics(y_test, y_test_mlp),
                'oof_mae': mae_mlp_oof},
        'gbr': {'train': calc_metrics(y_train, y_train_gbr), 'test': calc_metrics(y_test, y_test_gbr),
                'oof_mae': mae_gbr_oof},
        'rf': {'train': calc_metrics(y_train, y_train_rf), 'test': calc_metrics(y_test, y_test_rf),
               'oof_mae': mae_rf_oof},
        'blend': {'train': calc_metrics(y_train, y_train_blend), 'test': calc_metrics(y_test, y_test_blend)},
        'pesos': {'mlp': pesos[0], 'gbr': pesos[1], 'rf': pesos[2]},
    }

    predicciones = {
        'y_train_blend': y_train_blend, 'y_test_blend': y_test_blend,
        'y_train_mlp': y_train_mlp, 'y_test_mlp': y_test_mlp,
        'y_train_gbr': y_train_gbr, 'y_test_gbr': y_test_gbr,
        'y_train_rf': y_train_rf, 'y_test_rf': y_test_rf,
    }

    modelos_entrenados = {'gbr': gbr, 'rf': rf}

    return resultados, predicciones, modelos_entrenados


# ============================================================
# 6. DASHBOARD HTML (Plotly, interactivo)
# ============================================================
def construir_dashboard(fechas_train, y_train, fechas_test, y_test, predicciones,
                        resultados, features, timestamp):
    y_train_blend = predicciones['y_train_blend']
    y_test_blend = predicciones['y_test_blend']

    metrics_test = resultados['blend']['test']
    metrics_train = resultados['blend']['train']
    mae_prom = (metrics_train['mae'] + metrics_test['mae']) / 2

    # --- KPI cards (HTML/CSS) ---
    def kpi_card(label, valor, meta_ok=None):
        color = "#2e7d32" if (meta_ok is True) else ("#c62828" if meta_ok is False else "#1565c0")
        return f"""
        <div class="kpi">
            <div class="kpi-label">{label}</div>
            <div class="kpi-value" style="color:{color}">{valor}</div>
        </div>"""

    kpis_html = "".join([
        kpi_card("MAE Train", f"{metrics_train['mae']:.3f}", metrics_train['mae'] < OBJETIVO_MAE),
        kpi_card("MAE Test", f"{metrics_test['mae']:.3f}", metrics_test['mae'] < OBJETIVO_MAE),
        kpi_card("MAE Promedio (Train+Test)/2", f"{mae_prom:.3f}", mae_prom < OBJETIVO_MAE),
        kpi_card("R2 Test", f"{metrics_test['r2']:.3f}"),
        kpi_card("RMSE Test", f"{metrics_test['rmse']:.3f}"),
    ])

    # --- Figura 1: serie temporal completa ---
    fig_serie = go.Figure()
    fig_serie.add_trace(go.Scatter(x=fechas_train, y=y_train, name='Real (train)',
                                    line=dict(color='#90a4ae')))
    fig_serie.add_trace(go.Scatter(x=fechas_train, y=y_train_blend, name='Predicho (train)',
                                    line=dict(color='#42a5f5', dash='dot')))
    fig_serie.add_trace(go.Scatter(x=fechas_test, y=y_test, name='Real (test)',
                                    line=dict(color='#212121', width=3)))
    fig_serie.add_trace(go.Scatter(x=fechas_test, y=y_test_blend, name='Predicho (test)',
                                    line=dict(color='#e53935', width=3)))
    fig_serie.update_layout(title="Serie temporal: casos reales vs predichos (blend)",
                             xaxis_title="Fecha", yaxis_title="Casos de dengue",
                             template="plotly_white", height=420,
                             legend=dict(orientation="h", y=-0.2))

    # --- Figura 2: dispersion real vs predicho (test) ---
    max_val = max(y_test.max(), y_test_blend.max())
    fig_scatter = go.Figure()
    fig_scatter.add_trace(go.Scatter(x=y_test, y=y_test_blend, mode='markers',
                                      marker=dict(color='#5e35b1', size=8, opacity=0.7),
                                      name='Test'))
    fig_scatter.add_trace(go.Scatter(x=[0, max_val], y=[0, max_val], mode='lines',
                                      line=dict(color='black', dash='dash'), name='Perfecto'))
    fig_scatter.update_layout(title=f"Real vs Predicho - Test (R2={metrics_test['r2']:.3f})",
                               xaxis_title="Real", yaxis_title="Predicho",
                               template="plotly_white", height=420)

    # --- Figura 3: histograma de residuos ---
    errores = y_test - y_test_blend
    fig_res = go.Figure()
    fig_res.add_trace(go.Histogram(x=errores, marker_color='#00897b', nbinsx=25))
    fig_res.add_vline(x=0, line_dash="dash", line_color="red")
    fig_res.update_layout(title=f"Distribucion de errores - Test (MAE={metrics_test['mae']:.3f})",
                           xaxis_title="Error (Real - Predicho)", yaxis_title="Frecuencia",
                           template="plotly_white", height=420)

    # --- Figura 4: comparacion de MAE por modelo ---
    modelos_nombres = ['MLP (ensemble)', 'Gradient Boosting', 'Random Forest', 'Blend ponderado']
    maes_test_modelos = [resultados['mlp']['test']['mae'], resultados['gbr']['test']['mae'],
                          resultados['rf']['test']['mae'], resultados['blend']['test']['mae']]
    fig_comp = go.Figure()
    fig_comp.add_trace(go.Bar(x=modelos_nombres, y=maes_test_modelos,
                               marker_color=['#90caf9', '#a5d6a7', '#ffcc80', '#ef5350']))
    fig_comp.add_hline(y=OBJETIVO_MAE, line_dash="dash", line_color="green",
                        annotation_text=f"Objetivo MAE={OBJETIVO_MAE}")
    fig_comp.update_layout(title="MAE en Test por modelo", yaxis_title="MAE",
                            template="plotly_white", height=420)

    # --- Figura 5: importancia de variables (Random Forest) ---
    rf_model = None
    importancias = None
    return fig_serie, fig_scatter, fig_res, fig_comp, kpis_html, mae_prom


def guardar_dashboard_html(fig_serie, fig_scatter, fig_res, fig_comp, kpis_html,
                           mae_prom, importancia_html, timestamp):
    archivo_html = os.path.join(RUTA_RESULTADOS, f'dashboard_desempeno_{timestamp}.html')

    objetivo_ok = mae_prom < OBJETIVO_MAE
    banner_color = "#2e7d32" if objetivo_ok else "#c62828"
    banner_texto = (f"OBJETIVO LOGRADO: MAE promedio = {mae_prom:.3f} (< {OBJETIVO_MAE})"
                    if objetivo_ok else
                    f"MAE promedio = {mae_prom:.3f} (objetivo: < {OBJETIVO_MAE})")

    html = f"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<title>Dashboard de Desempeno - Prediccion de Dengue</title>
<style>
  body {{ font-family: 'Segoe UI', Arial, sans-serif; background:#f5f6fa; margin:0; padding:24px; color:#212121; }}
  h1 {{ font-size:22px; margin-bottom:4px; }}
  .subt {{ color:#616161; margin-top:0; margin-bottom:20px; }}
  .banner {{ background:{banner_color}; color:white; padding:14px 20px; border-radius:10px;
             font-weight:600; margin-bottom:24px; }}
  .kpis {{ display:flex; gap:16px; flex-wrap:wrap; margin-bottom:28px; }}
  .kpi {{ background:white; border-radius:12px; padding:16px 22px; box-shadow:0 1px 4px rgba(0,0,0,0.08);
          min-width:150px; }}
  .kpi-label {{ font-size:12px; color:#757575; text-transform:uppercase; letter-spacing:0.03em; }}
  .kpi-value {{ font-size:26px; font-weight:700; margin-top:4px; }}
  .grid {{ display:grid; grid-template-columns:1fr 1fr; gap:20px; }}
  .card {{ background:white; border-radius:12px; padding:8px; box-shadow:0 1px 4px rgba(0,0,0,0.08); }}
  .full {{ grid-column: 1 / -1; }}
  @media (max-width:900px) {{ .grid {{ grid-template-columns:1fr; }} }}
</style>
</head>
<body>
  <h1>Dashboard de Desempeno del Modelo - Prediccion de Casos de Dengue</h1>
  <p class="subt">Generado el {datetime.now().strftime('%d/%m/%Y %H:%M')} | Features: {len(FEATURES_FIJAS)} | Modelo: Blend (MLP ensemble + Gradient Boosting + Random Forest)</p>

  <div class="banner">{banner_texto}</div>

  <div class="kpis">
    {kpis_html}
  </div>

  <div class="grid">
    <div class="card full">{fig_serie.to_html(full_html=False, include_plotlyjs='cdn')}</div>
    <div class="card">{fig_scatter.to_html(full_html=False, include_plotlyjs=False)}</div>
    <div class="card">{fig_res.to_html(full_html=False, include_plotlyjs=False)}</div>
    <div class="card full">{fig_comp.to_html(full_html=False, include_plotlyjs=False)}</div>
    {f'<div class="card full">{importancia_html}</div>' if importancia_html else ''}
  </div>
</body>
</html>
"""
    with open(archivo_html, 'w', encoding='utf-8') as f:
        f.write(html)

    print(f"\nDashboard guardado: {archivo_html}")
    return archivo_html


# ============================================================
# 7. GUARDAR PREDICCIONES Y METRICAS EN EXCEL
# ============================================================
def guardar_resultados_excel(fechas_test, y_test, predicciones, resultados, features, timestamp):
    archivo_pred = os.path.join(RUTA_RESULTADOS, f'predicciones_{timestamp}.xlsx')
    df_pred = pd.DataFrame({
        'fecha': fechas_test.values,
        'casos_reales': y_test,
        'pred_blend': predicciones['y_test_blend'],
        'pred_mlp': predicciones['y_test_mlp'],
        'pred_gbr': predicciones['y_test_gbr'],
        'pred_rf': predicciones['y_test_rf'],
        'error_blend': y_test - predicciones['y_test_blend'],
    })
    df_pred.to_excel(archivo_pred, index=False)

    filas = []
    for modelo in ['mlp', 'gbr', 'rf', 'blend']:
        filas.append({
            'modelo': modelo,
            'mae_train': resultados[modelo]['train']['mae'],
            'mae_test': resultados[modelo]['test']['mae'],
            'rmse_train': resultados[modelo]['train']['rmse'],
            'rmse_test': resultados[modelo]['test']['rmse'],
            'r2_train': resultados[modelo]['train']['r2'],
            'r2_test': resultados[modelo]['test']['r2'],
        })
    df_metrics = pd.DataFrame(filas)
    archivo_metrics = os.path.join(RUTA_RESULTADOS, f'metricas_{timestamp}.xlsx')
    df_metrics.to_excel(archivo_metrics, index=False)

    print(f"Predicciones guardadas: {archivo_pred}")
    print(f"Metricas guardadas: {archivo_metrics}")
    return archivo_pred, archivo_metrics


# ============================================================
# 8. EJECUCION PRINCIPAL
# ============================================================
if __name__ == "__main__":
    print("=" * 70)
    print("SISTEMA BLEND (MLP + GBR + RF) PARA MAE PROMEDIO < 3")
    print(f"ARCHIVO: {ARCHIVO_DATOS}")
    print("=" * 70)

    df = cargar_datos_reducidos()
    X_train, X_test, y_train, y_test, fechas_train, fechas_test = preparar_datos(df, train_ratio=0.8)

    if len(X_train) == 0 or len(X_test) == 0:
        raise ValueError("No hay suficientes datos para entrenar y probar")

    # 1) Buscar hiperparametros de la MLP con validacion temporal
    mejores_params = buscar_hiperparametros_mlp(X_train, y_train, n_iter=15, n_splits=4)

    # 2) Entrenar sistema blend ponderado
    resultados, predicciones, modelos_entrenados = entrenar_sistema_blend(
        X_train, y_train, X_test, y_test, mejores_params, n_mlp=15
    )

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    print(f"\n{'='*70}\nRESUMEN DE DESEMPENO\n{'='*70}")
    for modelo in ['mlp', 'gbr', 'rf', 'blend']:
        m = resultados[modelo]
        print(f"  {modelo.upper():6s} -> MAE train={m['train']['mae']:.3f} | "
              f"MAE test={m['test']['mae']:.3f} | R2 test={m['test']['r2']:.3f}")

    mae_prom = (resultados['blend']['train']['mae'] + resultados['blend']['test']['mae']) / 2
    print(f"\n  MAE PROMEDIO (train+test)/2 = {mae_prom:.3f}  (objetivo < {OBJETIVO_MAE})")
    if mae_prom < OBJETIVO_MAE:
        print("  OBJETIVO LOGRADO")
    else:
        print("  Objetivo aun no alcanzado. Sugerencias:")
        print("   - Aumentar n_mlp (ensemble) a 20-25")
        print("   - Ampliar n_iter en la busqueda de hiperparametros (20-30)")
        print("   - Revisar si hay fugas/duplicados en las features de RFE")
        print("   - Considerar agregar mas features de autocorrelacion (lags cortos)")

    # 3) Importancia de variables (Random Forest, sobre train escalado)
    rf_model = modelos_entrenados['rf']
    importancias = rf_model.feature_importances_
    orden = np.argsort(importancias)[::-1]
    fig_imp = go.Figure(go.Bar(
        x=[importancias[i] for i in orden],
        y=[FEATURES_FIJAS[i] for i in orden],
        orientation='h', marker_color='#7e57c2'
    ))
    fig_imp.update_layout(title="Importancia de variables (Random Forest)",
                           template="plotly_white", height=500,
                           yaxis=dict(autorange="reversed"))
    importancia_html = fig_imp.to_html(full_html=False, include_plotlyjs=False)

    # 4) Construir figuras del dashboard
    fig_serie, fig_scatter, fig_res, fig_comp, kpis_html, mae_prom = construir_dashboard(
        fechas_train, y_train, fechas_test, y_test, predicciones, resultados, FEATURES_FIJAS, timestamp
    )

    # 5) Guardar dashboard HTML
    archivo_dashboard = guardar_dashboard_html(
        fig_serie, fig_scatter, fig_res, fig_comp, kpis_html, mae_prom, importancia_html, timestamp
    )

    # 6) Guardar predicciones y metricas en Excel
    guardar_resultados_excel(fechas_test, y_test, predicciones, resultados, FEATURES_FIJAS, timestamp)

    print(f"\nResultados guardados en: {RUTA_RESULTADOS}")
    print(f"Abre el dashboard en tu navegador: {archivo_dashboard}")
    print("=" * 70)


SISTEMA BLEND (MLP + GBR + RF) PARA MAE PROMEDIO < 3
ARCHIVO: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\2_datos\2_procesados\dengue_reducido_rfe_n_20_20260810_164451.xlsx


FileNotFoundError: No se encontro el archivo especificado: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\2_datos\2_procesados\dengue_reducido_rfe_n_20_20260810_164451.xlsx